# DASHBOARD V1 


In [2]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

In [5]:
retail_d1 = pd.read_parquet(
    "../data/processed/online_retail_II_cleaned.parquet"
)
print("Rows:", len(retail_d1))
print("Columns:", len(retail_d1.columns))

retail_d1.head()

Rows: 1027614
Columns: 15


,Invoice,StockCode,Description,Description_clean,Quantity,InvoiceDate,Price,Price_clean,Customer_ID,Country,is_cancelled,is_stock_adjustment,original_was_cancelled,exclude_from_demand,is_price_outlier
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,6.95,13085,United Kingdom,False,False,False,False,False
1,489434,79323P,PINK CHERRY LIGHTS,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,6.75,13085,United Kingdom,False,False,False,False,False
2,489434,79323W,WHITE CHERRY LIGHTS,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,6.75,13085,United Kingdom,False,False,False,False,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE","RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,2.10,13085,United Kingdom,False,False,False,False,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,1.25,13085,United Kingdom,False,False,False,False,False


In [6]:
retail_d1.info()
retail_d1.shape

<class 'pandas.DataFrame'>
RangeIndex: 1027614 entries, 0 to 1027613
Data columns (total 15 columns):
 #   Column                  Non-Null Count    Dtype         
---  ------                  --------------    -----         
 0   Invoice                 1027614 non-null  str           
 1   StockCode               1027614 non-null  str           
 2   Description             1023349 non-null  str           
 3   Description_clean       1027238 non-null  str           
 4   Quantity                1027614 non-null  int64         
 5   InvoiceDate             1027614 non-null  datetime64[us]
 6   Price                   1027614 non-null  float64       
 7   Price_clean             1027614 non-null  float64       
 8   Customer_ID             794498 non-null   Int64         
 9   Country                 1027614 non-null  str           
 10  is_cancelled            1027614 non-null  bool          
 11  is_stock_adjustment     1027614 non-null  bool          
 12  original_was_cancelled  1

(1027614, 15)

In [7]:
dashboard = retail_d1.copy()

In [8]:
print(retail_d1.shape)
print(retail_d1.columns.tolist())
retail_d1.head()


(1027614, 15)
['Invoice', 'StockCode', 'Description', 'Description_clean', 'Quantity', 'InvoiceDate', 'Price', 'Price_clean', 'Customer_ID', 'Country', 'is_cancelled', 'is_stock_adjustment', 'original_was_cancelled', 'exclude_from_demand', 'is_price_outlier']


,Invoice,StockCode,Description,Description_clean,Quantity,InvoiceDate,Price,Price_clean,Customer_ID,Country,is_cancelled,is_stock_adjustment,original_was_cancelled,exclude_from_demand,is_price_outlier
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,6.95,13085,United Kingdom,False,False,False,False,False
1,489434,79323P,PINK CHERRY LIGHTS,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,6.75,13085,United Kingdom,False,False,False,False,False
2,489434,79323W,WHITE CHERRY LIGHTS,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,6.75,13085,United Kingdom,False,False,False,False,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE","RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,2.10,13085,United Kingdom,False,False,False,False,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,1.25,13085,United Kingdom,False,False,False,False,False


## DASHBOARD V1 - SALES DEFINITIONS

In [9]:
# Transaction-level revenue
dashboard["Revenue"] = (
    dashboard["Quantity"] * dashboard["Price_clean"]
)

In [10]:
dashboard["Is_Valid_Sale"] = (
    (dashboard["is_cancelled"] == False) &
    (dashboard["is_stock_adjustment"] == False) &
    (dashboard["Quantity"] > 0) &
    (dashboard["Price_clean"] > 0)
)

In [11]:
# Revenue from valid sales only
dashboard["Sales_Revenue"] = dashboard["Revenue"].where(
    dashboard["Is_Valid_Sale"],
    0
)

# Units sold from valid sales only
dashboard["Sales_Quantity"] = dashboard["Quantity"].where(
    dashboard["Is_Valid_Sale"],
    0
)

print("Dashboard rows:", len(dashboard))
print("Valid sales rows:", dashboard["Is_Valid_Sale"].sum())
print("Valid sales revenue:", dashboard["Sales_Revenue"].sum())
print("Units sold:", dashboard["Sales_Quantity"].sum())

Dashboard rows: 1027614
Valid sales rows: 1003701
Valid sales revenue: 19605329.597000003
Units sold: 11188409


## Time Dimensions

In [12]:
dashboard["Year"] = dashboard["InvoiceDate"].dt.year
dashboard["Month"] = dashboard["InvoiceDate"].dt.month

dashboard["Month_Name"] = dashboard["InvoiceDate"].dt.strftime("%b")

dashboard["Year_Month"] = (
    dashboard["InvoiceDate"]
    .dt.to_period("M")
    .astype(str)
)

dashboard["Year_Month_Date"] = (
    dashboard["InvoiceDate"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

dashboard[[
    "InvoiceDate",
    "Year",
    "Month",
    "Month_Name",
    "Year_Month"
]].head()

,InvoiceDate,Year,Month,Month_Name,Year_Month
0,2009-12-01 07:45:00,2009,12,Dec,2009-12
1,2009-12-01 07:45:00,2009,12,Dec,2009-12
2,2009-12-01 07:45:00,2009,12,Dec,2009-12
3,2009-12-01 07:45:00,2009,12,Dec,2009-12
4,2009-12-01 07:45:00,2009,12,Dec,2009-12


## DASHBOARD VALIDATION

In [13]:
print("Rows:", len(dashboard))

print(
    "Unique orders:",
    dashboard.loc[
        dashboard["Is_Valid_Sale"],
        "Invoice"
    ].nunique()
)

print(
    "Unique customers:",
    dashboard.loc[
        dashboard["Is_Valid_Sale"] &
        dashboard["Customer_ID"].notna(),
        "Customer_ID"
    ].nunique()
)

print(
    "Unique products:",
    dashboard.loc[
        dashboard["Is_Valid_Sale"],
        "StockCode"
    ].nunique()
)

print(
    "Countries:",
    dashboard.loc[
        dashboard["Is_Valid_Sale"],
        "Country"
    ].nunique()
)

Rows: 1027614
Unique orders: 39523
Unique customers: 5853
Unique products: 4904
Countries: 43


## DASHBOARD V1 DATASET

In [14]:
dashboard_path = "../data/processed/retail_dashboard_v1.parquet"

dashboard.to_parquet(
    dashboard_path,
    index=False
)

print("Saved:", dashboard_path)

Saved: ../data/processed/retail_dashboard_v1.parquet


In [15]:
from pathlib import Path

print(Path(dashboard_path).resolve())

C:\Users\DELL\Desktop\Retail-Demand-and-Market-Intelligence-Analytics\data\processed\retail_dashboard_v1.parquet
